# Naver 뉴스 본문 수집 (Selenium Colab용)

URL 수집 노트북에서 만든 `링크_*.json` 파일을 읽어 기사 제목, 본문, 날짜, 카테고리를 수집한다. 로컬용 Selenium 노트북과 수집 로직은 같고, Colab 실행을 위해 Google Chrome 설치와 Google Drive 마운트 셀이 포함되어 있다.

- 입력: `data/링크_{query}_{YYMMDD}_{YYMMDD}.json`
- 출력: `data/본문_{query}_{YYMMDD}_{YYMMDD}.csv`
- 보조 출력: 중간 재개용 체크포인트 JSON, 재실패 URL JSON
- 특징: 월 단위 자동 실행, 중간 재개, 오류 URL 1회 재시도, 완료 파일 건너뛰기


In [ ]:
# Colab 환경 세팅 — Selenium, Google Chrome 설치
!wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q /tmp/google-chrome.deb
!pip install -q selenium


In [ ]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Drive 안의 프로젝트 폴더로 이동
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
from datetime import datetime
import pandas as pd
import json
import os
import shutil
import subprocess
import calendar
import random

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 키워드와 수집할 시작/끝 년월 지정
# query 하나당 start_ym부터 end_ym까지 월 단위 작업으로 자동 분할, start_ym/end_ym 형식: 'YYYY.MM'
# 같은 carrier여도 SKT/SK텔레콤, LG U+/LG유플러스는 별 query로 처리해 별 파일로 저장
query_ranges = [
    {'query': 'SK텔레콤', 'start_ym': '2025.04', 'end_ym': '2025.08'},
    # {'query': 'KT', 'start_ym': '2025.09', 'end_ym': '2026.01'},
    # {'query': 'LG유플러스', 'start_ym': '2025.08', 'end_ym': '2025.12'},
    # {'query': 'SKT', 'start_ym': '2025.04', 'end_ym': '2025.08'},
    # {'query': 'LG U+', 'start_ym': '2025.08', 'end_ym': '2025.12'},
]

# 월 단위 작업 목록 생성
# 월 마지막 일자는 calendar.monthrange로 자동 계산하므로 28/29/30/31일을 직접 입력하지 않아도 됨
def build_monthly_jobs(query_ranges):
    jobs = []

    for item in query_ranges:
        query = item['query']
        start_year, start_month = map(int, item['start_ym'].split('.'))
        end_year, end_month = map(int, item['end_ym'].split('.'))

        # 시작 년월이 끝 년월보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if (start_year, start_month) > (end_year, end_month):
            raise ValueError(f"시작 년월이 끝 년월보다 늦습니다: {item}")

        # 시작 월부터 끝 월까지 한 달씩 이동하면서 작업 생성
        year, month = start_year, start_month
        while (year, month) <= (end_year, end_month):
            # 해당 월의 마지막 날짜 자동 계산 (윤년 2월 29일 포함)
            last_day = calendar.monthrange(year, month)[1]
            jobs.append({
                'query': query,
                'start_date': f'{year}.{month:02d}.01',
                'end_date': f'{year}.{month:02d}.{last_day:02d}',
            })

            # 다음 달로 이동, 12월 다음은 다음 해 1월로 변경
            month += 1
            if month == 13:
                year += 1
                month = 1

    return jobs

# 변수 정의 (날짜 형식: 'YYYY.MM.DD')
# 생성된 jobs는 셀 5에서 순서대로 실행
jobs = build_monthly_jobs(query_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'

# 저장할 폴더 지정
# 링크 파일, 체크포인트, 본문 CSV, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = os.path.join(PROJECT_DIR, 'notebook', 'crawling', 'data')
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

options = Options()
options.add_argument(f'user-agent={USER_AGENT}')  # 요청 환경을 일정하게 유지하기 위해 User-Agent 고정
print(f'User-Agent: {USER_AGENT}')
options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동화 제어 관련 switch 제외
options.add_experimental_option('useAutomationExtension', False)  # Selenium 자동화 확장 비활성화
options.add_argument('--disable-blink-features=AutomationControlled')  # AutomationControlled 플래그 비활성화
options.add_argument('--headless=new')  # Colab은 GUI가 없으므로 새 headless 모드 사용
options.add_argument('--no-sandbox')  # Colab 컨테이너 환경에서 Chrome 실행 안정화
options.add_argument('--disable-dev-shm-usage')  # /dev/shm 용량 부족으로 Chrome이 죽는 문제 완화
options.add_argument('--disable-gpu')  # headless 환경에서 GPU 관련 오류 방지
options.add_argument('--window-size=1920,1080')  # headless에서도 일정한 화면 크기로 렌더링

# Colab chromium-browser 패키지는 snap 래퍼라 Selenium에서 자주 실패
# 설치 셀에서 받은 Google Chrome 사용, ChromeDriver는 Selenium Manager에 맡김
chrome_binary = shutil.which('google-chrome') or shutil.which('google-chrome-stable') or '/usr/bin/google-chrome'
if not os.path.exists(chrome_binary):
    raise FileNotFoundError('Google Chrome을 찾지 못했습니다. 설치 셀을 먼저 다시 실행해 주세요.')

options.binary_location = chrome_binary
print(f'Chrome binary: {chrome_binary}')
subprocess.run([chrome_binary, '--version'], check=False)

# Chrome 드라이버 설정
# Selenium Manager가 현재 Chrome 버전에 맞는 ChromeDriver를 자동으로 찾거나 내려받음
service = Service()
driver = webdriver.Chrome(service=service, options=options)

# navigator.webdriver 플래그 제거 — 자동화 탐지를 회피하기 위해 새 페이지 진입 시마다 주입
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})


In [ ]:
import random

# 서버 부담을 줄이기 위해 기사/job 사이에 짧은 랜덤 대기
ARTICLE_PAUSE_RANGE_SEC = (0.8, 2.0)
JOB_PAUSE_RANGE_SEC = (10, 25)
# 체크포인트 저장 간격 — N건마다 한 번씩 중간저장 파일 갱신
CHECKPOINT_INTERVAL = 100
SKIP_COMPLETED = True

# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


# 파일명에 들어가는 기간 접미사 만들기 (예: 2025.04.01 ~ 2025.04.30 -> 250401_250430)
def make_period_suffix(start_date, end_date):
    return f"{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}"


# 기사 한 건에서 title/body/pubdate/category 추출
# title/body/pubdate 중 하나라도 비면 호출부에서 err_idx로 기록할 수 있도록 ValueError 발생
def extract_article(driver, link):
    # 실제 네이버 뉴스 웹페이지로 이동
    driver.get(link)

    # 페이지 로딩 대기
    time.sleep(0.6)

    # 제목 추출하기
    title = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
    title = title[0].text if title else ''

    # 본문 추출하기 — 줄바꿈 제거해서 단일 문자열로 정리
    body = driver.find_elements(By.ID, 'newsct_article')
    body = body[0].text.replace('\n', '') if body else ''

    # 날짜 추출하기 — 사람이 읽는 라벨이 아니라 data-date-time 속성값을 사용 (ISO 포맷)
    pubdate_element = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
    pubdate = pubdate_element[0].get_attribute('data-date-time') if pubdate_element else ''

    # 카테고리 추출하기 — 페이지 상단 카테고리 탭 중 현재 활성화된(aria-selected="true") 항목
    category_element = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')
    category = category_element[0].text if category_element else ''

    # 제목/본문/날짜 중 하나라도 없으면 실패 — 호출부에서 err_idx로 기록
    if not title or not body or not pubdate:
        raise ValueError('title/body/pubdate 중 일부를 추출하지 못함')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 추출한 링크에 직접 방문하여 크롤링 진행
# 체크포인트 기반 재개 + 오류 자동 1회 재시도 + 재실패 URL JSON 저장
def collect_bodies(query, start_date, end_date, save_dir=SAVE_DIR, driver=driver):
    # 파일명 키로 쓸 기간 접미사 (예: 250401_250430)
    period = make_period_suffix(start_date, end_date)
    # 입력: URL 수집 단계가 만들어 둔 월별 링크 JSON
    links_file_name = f"링크_{query}_{period}.json"
    links_path = os.path.join(save_dir, links_file_name)
    # 체크포인트: 중간 결과(all_results) + 오류 인덱스 + 다음 인덱스를 보관
    checkpoint_name = f"체크포인트_본문_{query}_{period}.json"
    checkpoint_path = os.path.join(save_dir, checkpoint_name)
    # 최종 출력: 월별 본문 CSV
    csv_file_name = f"본문_{query}_{period}.csv"
    csv_save_path = os.path.join(save_dir, csv_file_name)

    # 최종 파일이 이미 있으면 같은 월은 건너뜀 (재실행 시 idempotent)
    if SKIP_COMPLETED and os.path.exists(csv_save_path):
        print()
        print(f"=== {query} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {csv_save_path}")
        return csv_save_path

    # 월별 링크 파일 불러오기
    if os.path.exists(links_path):
        with open(links_path, 'r', encoding='utf-8') as f:
            naver_news_links = json.load(f)
        # 기사 한 건당 약 1.4초 가정한 거친 예상치 — 로그 확인용
        est_min = len(naver_news_links) * 1.4 / 60
        print()
        print(f"=== {query} / {start_date} ~ {end_date} 본문 수집 시작 ===")
        print(f'링크 {len(naver_news_links)}개 불러옴: {links_path}')
        print(f'예상 소요 시간: 약 {est_min:.0f}분')
    else:
        raise FileNotFoundError(f'링크 파일 없음 — url_수집_colab.ipynb를 먼저 실행하세요\n경로: {links_path}')

    # 이전에 중단된 작업이 있으면 이어받기 — next_i 인덱스 다음부터 시작
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # JSON은 dict 키를 문자열로 저장하므로 int로 다시 변환
        all_results = {int(k): v for k, v in checkpoint.get('all_results', {}).items()}
        err_idx = checkpoint.get('err_idx', [])
        i = checkpoint.get('next_i', 0)
        print(f'체크포인트 발견 — {i}번째부터 이어서 시작 (이미 수집: {len(all_results)}건)')
    else:
        all_results = dict()
        i = 0
        err_idx = []
        print('새로 시작')

    # 본문 수집 메인 루프 — i 인덱스를 함께 들고 다녀 체크포인트와 동기화
    for link in naver_news_links[i:]:
        try:
            all_results[i] = extract_article(driver, link)

            # 진행 상황 확인용 코드
            print(f'[{i+1} / {len(naver_news_links)}] 	 {(i+1)/len(naver_news_links)*100:.2f}% 	 error: {len(err_idx)}')

            i += 1

            # 중간저장 — N건마다 체크포인트 갱신해 중간 중단에도 진행 보존
            if i % CHECKPOINT_INTERVAL == 0:
                with open(checkpoint_path, 'w', encoding='utf-8') as f:
                    json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)
                print(f'체크포인트 저장 — {i}건 완료')

            polite_sleep('다음 기사 전', ARTICLE_PAUSE_RANGE_SEC)

        except Exception as exc:
            # 오류 인덱스를 err_idx에 누적해두고 즉시 체크포인트도 갱신
            print(f'오류 발생 — index {i}: {exc!r}')
            err_idx.append(i)
            i += 1
            with open(checkpoint_path, 'w', encoding='utf-8') as f:
                json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)

    # 1차 오류 자동 재시도 — 페이지가 늦게 떴거나 일시적 네트워크 문제였던 경우를 한 번 더 시도
    if err_idx:
        print()
        print(f'오류 {len(err_idx)}건 재시도 시작...')
        re_err_idx = []

        for retry_i in err_idx:
            try:
                link = naver_news_links[retry_i]
                all_results[retry_i] = extract_article(driver, link)
                print(f'재시도 성공 — index {retry_i}')
                polite_sleep('다음 재시도 전', ARTICLE_PAUSE_RANGE_SEC)
            except Exception as exc:
                print(f'재시도 실패 — index {retry_i}: {exc!r}')
                re_err_idx.append(retry_i)

        # 재시도 후에도 실패한 인덱스만 남김 — 본문_재실패_*.json으로 저장 대상
        err_idx = re_err_idx
        print(f'재시도 완료 — 재실패: {len(err_idx)}건')

    # 수집한 정보들을 dataframe으로 변환
    df = pd.DataFrame(all_results).T

    if df.empty:
        raise ValueError('수집된 본문 데이터가 없습니다.')

    # 수집한 기사들 중 중복인 경우 이를 제거
    df_no_duplicates = df.drop_duplicates().reset_index(drop=True)

    # 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
    df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'], errors='coerce')
    df_sorted = df_no_duplicates.sort_values(by='pubdate')

    # 수집한 정보들을 csv로 저장 (Google Drive에 저장)
    df_sorted.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {csv_save_path}')

    if err_idx:
        # 재시도 후에도 실패한 URL은 별 JSON으로 떨어뜨려 Selenium 재시도 노트북에서 다시 시도 가능
        failed_path = os.path.join(save_dir, f"본문_재실패_{query}_{period}.json")
        with open(failed_path, 'w', encoding='utf-8') as f:
            json.dump({'err_idx': err_idx, 'links': [naver_news_links[x] for x in err_idx]}, f, ensure_ascii=False, indent=2)
        print(f'재실패 목록 저장: {failed_path}')
    elif os.path.exists(checkpoint_path):
        # 재실패가 없을 때만 체크포인트 삭제 — 실패가 남아있으면 디버깅용으로 보존
        os.remove(checkpoint_path)

    print(f'본문 수집 완료 — 총 {len(df_sorted)}건 / 오류 {len(err_idx)}건')
    return csv_save_path


# 생성된 jobs를 순서대로 실행
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # job 딕셔너리의 query/start_date/end_date를 collect_bodies 인자로 전달
        results.append(collect_bodies(**job))
    except Exception as exc:
        # 한 월에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어감: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 월별 job으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = os.path.join(SAVE_DIR, '본문_수집실패목록.json')
    with open(failures_path, 'w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)


In [ ]:
# 브라우저 창 닫기
driver.quit()
